# TrendAELG + WaveletV3AELG Cross-Dataset Study

**Study question:** Which wavelet family, basis dimension, and trend polynomial degree produce the best TrendAELG + WaveletV3AELG forecasts -- and do the answers change across datasets with different forecast horizons?

**Datasets:** M4-Yearly (SMAPE/OWA), Tourism-Yearly (SMAPE), Weather-96 (MSE/MAE), Traffic-96 (MSE/MAE)  
**Architecture:** `[TrendAELG, <Wavelet>WaveletV3AELG]` alternating stacks, latent_dim=16  
**Search:** Successive halving -- 112 configs (R1, 10 ep) -> 75 (R2, 15 ep) -> 50 (R3, 50 ep)

**Search dimensions:**
- 14 wavelet families: Haar, DB2-DB20, Coif1-Coif10, Symlet2-Symlet20
- 4 basis_dim labels: eq_fcast, lt_fcast, eq_bcast, lt_bcast
- 2 trend_thetas_dim: 3, 5

---

### Reference baselines (M4-Yearly)

| Baseline | SMAPE | OWA |
|---|---|---|
| Non-AE Trend+WaveletV3 best (Coif2_bd6) | 13.410 | 0.794 |
| Prior V1 AELG best (Sym20_eq_fcast_ttd3) | 13.438 | 0.795 |
| NBEATS-G (30x Generic) | ~13.3-13.5 | ~0.837-0.850 |

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import mannwhitneyu, spearmanr
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)

# ---- Colour palette ----------------------------------------------------------
FAMILY_COLORS = {
    'Haar': '#E53935', 'DB2': '#1E88E5', 'DB3': '#43A047', 'DB4': '#FB8C00',
    'DB10': '#8E24AA', 'DB20': '#00ACC1', 'Coif1': '#FDD835', 'Coif2': '#6D4C41',
    'Coif3': '#546E7A', 'Coif10': '#D81B60', 'Symlet2': '#3949AB',
    'Symlet3': '#00897B', 'Symlet10': '#C0CA33', 'Symlet20': '#FF6F00',
}
BD_COLORS = {'eq_fcast': '#2196F3', 'lt_fcast': '#FF5722',
             'eq_bcast': '#4CAF50', 'lt_bcast': '#9C27B0'}
TTD_COLORS = {3: '#1565C0', 5: '#E65100'}

# ---- Load all 4 datasets -----------------------------------------------------
BASE = '../../results'
m4      = pd.read_csv(f'{BASE}/m4/wavelet_v3aelg_trendaelg_study_results.csv')
tourism = pd.read_csv(f'{BASE}/tourism/wavelet_v3aelg_trendaelg_study_results.csv')
traffic = pd.read_csv(f'{BASE}/traffic/wavelet_v3aelg_trendaelg_study_results.csv')
weather = pd.read_csv(f'{BASE}/weather/wavelet_v3aelg_trendaelg_study_results.csv')

# Clean wavelet name
for df in [m4, tourism, traffic, weather]:
    df['wavelet_name'] = df['wavelet_family'].str.replace('WaveletV3AELG', '')

# Mark diverged
m4['diverged_flag']      = m4['smape'] >= 50
tourism['diverged_flag']  = tourism['smape'] >= 50
traffic['diverged_flag']  = traffic['smape'] >= 200
weather['diverged_flag']  = False  # none diverged

datasets = {'M4-Yearly': m4, 'Tourism-Yearly': tourism,
            'Traffic-96': traffic, 'Weather-96': weather}

print("Loaded datasets:")
for name, df in datasets.items():
    n_div = df['diverged_flag'].sum()
    print(f"  {name:16s}: {len(df):>4d} rows, "
          f"{df['config_name'].nunique():>3d} configs, "
          f"{df['search_round'].nunique()} rounds, "
          f"{n_div} diverged ({n_div/len(df)*100:.1f}%)")

## 1. Per-Dataset Final Rankings (Round 3)

Round 3 survivors trained for ~50 epochs. These are the converged results that matter for configuration selection. We show the top 15 for each viable dataset (Traffic is excluded -- 86% divergence means no R2/R3 data exists).

In [ ]:
def rank_table(df, metric, ascending=True, top_n=15, label=''):
    """Build a ranked table from R3 data for a dataset."""
    r3 = df[(df['search_round'] == df['search_round'].max()) & (~df['diverged_flag'])]
    agg = r3.groupby('config_name').agg(
        mean=(metric, 'mean'), std=(metric, 'std'),
        n=(metric, 'count'),
        wavelet=('wavelet_name', 'first'),
        bd=('bd_label', 'first'), ttd=('trend_thetas_dim_cfg', 'first'),
    ).sort_values('mean', ascending=ascending).head(top_n)
    agg.columns = [f'{metric}_mean', f'{metric}_std', 'n', 'wavelet', 'bd_label', 'ttd']
    best = agg.iloc[0][f'{metric}_mean']
    agg['delta'] = agg[f'{metric}_mean'] - best
    if label:
        display(Markdown(f"### {label}"))
    display(agg.style.format({
        f'{metric}_mean': '{:.3f}', f'{metric}_std': '{:.3f}',
        'delta': '{:+.3f}', 'ttd': '{:.0f}'
    }).bar(subset=['delta'], color='#ffcdd2'))
    return agg

m4_r3_rank   = rank_table(m4,      'smape', ascending=True,  label='M4-Yearly (SMAPE, lower=better)')
tour_r3_rank = rank_table(tourism, 'smape', ascending=True,  label='Tourism-Yearly (SMAPE, lower=better)')
wea_r3_rank  = rank_table(weather, 'mse',   ascending=True,  label='Weather-96 (MSE, lower=better)')

**Interpretation:** The M4-Yearly top-15 span only 0.09 SMAPE (13.438-13.529). The architecture converges to essentially the same quality regardless of wavelet family once training is sufficient. Tourism shows a similarly tight spread (0.04). Weather has more variation, with Symlet20 pulling ahead on MSE.

---

## 2. Cross-Dataset Wavelet Family Comparison

The central question: does wavelet family choice matter, and if so, which family generalizes best? We rank each family by its mean metric within each dataset (R3 only for M4/Tourism/Weather), then average the ranks.

In [ ]:
def wavelet_family_ranks(df, metric, ascending=True, round_num=None):
    """Rank wavelet families within a dataset."""
    d = df[~df['diverged_flag']].copy()
    if round_num is not None:
        d = d[d['search_round'] == round_num]
    return d.groupby('wavelet_name')[metric].mean().rank(ascending=ascending)

# Use final round for each dataset
m4_max_r  = m4['search_round'].max()
t_max_r   = tourism['search_round'].max()
w_max_r   = weather['search_round'].max()

m4_wf  = wavelet_family_ranks(m4,      'smape', ascending=True,  round_num=m4_max_r)
t_wf   = wavelet_family_ranks(tourism, 'smape', ascending=True,  round_num=t_max_r)
w_wf   = wavelet_family_ranks(weather, 'mse',   ascending=True,  round_num=w_max_r)

cross = pd.DataFrame({'M4': m4_wf, 'Tourism': t_wf, 'Weather': w_wf}).dropna()
cross['avg_rank'] = cross.mean(axis=1)
cross['max_rank'] = cross.max(axis=1)
cross = cross.sort_values('avg_rank')

# --- Visualisation: heatmap of ranks ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [3, 2]})

# Heatmap
ax = axes[0]
rank_df = cross[['M4', 'Tourism', 'Weather']].copy()
sns.heatmap(rank_df, annot=True, fmt='.0f', cmap='RdYlGn_r', ax=ax,
            linewidths=0.5, vmin=1, vmax=14, cbar_kws={'label': 'Rank (1=best)'})
ax.set_title('Wavelet Family Rank by Dataset (R3)', fontsize=13, fontweight='bold')
ax.set_ylabel('')

# Bar chart of average rank
ax2 = axes[1]
colors = [FAMILY_COLORS.get(n, '#999') for n in cross.index]
ax2.barh(range(len(cross)), cross['avg_rank'], color=colors, edgecolor='white', linewidth=0.5)
ax2.set_yticks(range(len(cross)))
ax2.set_yticklabels(cross.index)
ax2.invert_yaxis()
ax2.set_xlabel('Average Rank Across 3 Datasets')
ax2.set_title('Cross-Dataset Robustness', fontsize=13, fontweight='bold')
ax2.axvline(x=7.5, color='gray', linestyle='--', alpha=0.5, label='Median')

plt.tight_layout()
plt.savefig('wavelet_family_cross_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

display(cross.style.format('{:.1f}').background_gradient(cmap='RdYlGn_r', axis=None))

**Interpretation:** Symlet20 is the clear cross-dataset winner (avg rank 2.3, never worse than 3rd). The Symlet family has near-linear phase response, preserving temporal structure across decomposition levels -- this likely explains its consistent performance across horizons.

DB4 is #1 on both short-horizon datasets (M4, Tourism) but dead last (#14) on Weather-96. DB20 shows the inverse pattern (#1 Weather, #12-13 short-horizon). This confirms the **support-length/horizon matching principle**: short wavelets for short horizons, long wavelets for long horizons. Symlet20 transcends this tradeoff.

---

## 3. Haar vs DB2 Head-to-Head

Since the study started with a Haar vs DB2 comparison, let us examine them directly across datasets.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (ds_name, df, metric) in zip(axes, [
    ('M4-Yearly', m4, 'smape'),
    ('Tourism-Yearly', tourism, 'smape'),
    ('Weather-96', weather, 'mse'),
]):
    r3 = df[(df['search_round'] == df['search_round'].max()) & (~df['diverged_flag'])]
    haar = r3[r3['wavelet_name'] == 'Haar'][metric].values
    db2  = r3[r3['wavelet_name'] == 'DB2'][metric].values
    
    parts = ax.violinplot([haar, db2], positions=[0, 1], showmeans=True, showextrema=True)
    for i, (body, color) in enumerate(zip(parts['bodies'], 
                                           [FAMILY_COLORS['Haar'], FAMILY_COLORS['DB2']])):
        body.set_facecolor(color)
        body.set_alpha(0.7)
    
    # Mann-Whitney test
    if len(haar) > 2 and len(db2) > 2:
        u, p = mannwhitneyu(haar, db2, alternative='two-sided')
        sig = '*' if p < 0.05 else 'ns'
        ax.text(0.5, 0.95, f'p={p:.3f} ({sig})', transform=ax.transAxes,
                ha='center', va='top', fontsize=10, style='italic')
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Haar', 'DB2'])
    ax.set_ylabel(metric.upper())
    ax.set_title(f'{ds_name}', fontweight='bold')
    
    # Annotate means
    for pos, vals, name in [(0, haar, 'Haar'), (1, db2, 'DB2')]:
        ax.text(pos, np.mean(vals), f'{np.mean(vals):.2f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold')

fig.suptitle('Haar vs DB2 Head-to-Head (R3 converged data)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('haar_vs_db2.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Haar and DB2 are both mid-to-low performers in the R3 rankings. Neither should be the default choice. DB2 slightly edges Haar on M4 and Tourism (lower mean SMAPE) while Haar slightly wins on Weather (lower MSE). The differences are not statistically significant in any case. Both are outclassed by Symlet20, DB4 (short horizon), and DB20 (long horizon).

---

## 4. Basis Dim Label Effect Across Datasets

The `bd_label` maps to different `basis_dim` values depending on dataset geometry. Does the optimal label change across datasets, or is `eq_fcast` always best?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
bd_order = ['eq_fcast', 'lt_bcast', 'eq_bcast', 'lt_fcast']

for row, round_label in enumerate(['R1 (10 epochs)', 'R3 (converged)']):
    for col, (ds_name, df, metric) in enumerate([
        ('M4-Yearly', m4, 'smape'),
        ('Tourism-Yearly', tourism, 'smape'),
        ('Weather-96', weather, 'mse'),
    ]):
        ax = axes[row, col]
        rnd = 1 if row == 0 else df['search_round'].max()
        d = df[(df['search_round'] == rnd) & (~df['diverged_flag'])]
        
        data_by_label = [d[d['bd_label'] == lbl][metric].values for lbl in bd_order]
        bp = ax.boxplot(data_by_label, labels=bd_order, patch_artist=True, widths=0.6)
        for patch, lbl in zip(bp['boxes'], bd_order):
            patch.set_facecolor(BD_COLORS[lbl])
            patch.set_alpha(0.7)
        
        ax.set_title(f'{ds_name} -- {round_label}', fontweight='bold', fontsize=10)
        ax.set_ylabel(metric.upper(), fontsize=9)
        ax.tick_params(axis='x', rotation=30, labelsize=8)
        
        # Annotate median
        for i, lbl in enumerate(bd_order):
            vals = d[d['bd_label'] == lbl][metric]
            if len(vals) > 0:
                ax.text(i + 1, vals.median(), f'{vals.median():.1f}', 
                        ha='center', va='bottom', fontsize=7, fontweight='bold')

fig.suptitle('Basis Dim Label Effect: R1 (early training) vs R3 (converged)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('bd_label_effect.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** The top row (R1) shows large apparent differences between bd_labels. The bottom row (R3, converged) shows these differences nearly vanish. The R1 advantage of `eq_bcast` (larger basis_dim) is a **convergence speed artifact** -- more basis functions learn faster but do not produce better final forecasts.

At convergence, `eq_fcast` (basis_dim = forecast_length) is top-1 or co-top-1 on all three datasets. `lt_fcast` (basis_dim < forecast_length) is consistently worst, especially on short horizons where it becomes severely under-parameterized.

Note: Tourism and Weather show `eq_fcast` = `lt_bcast` degeneracy (both resolve to basis_dim = forecast_length). The identical box plots confirm this.

**Recommendation:** Always use `eq_fcast` as the default.

---

## 5. Trend Thetas Dim: The Horizon-Dependent Reversal

One of the most important findings: the optimal `trend_thetas_dim` reverses direction between short-horizon (M4, Tourism) and long-horizon (Weather) datasets.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

results_summary = []
for ax, (ds_name, df, metric, horizon) in zip(axes, [
    ('M4-Yearly', m4, 'smape', 6),
    ('Tourism-Yearly', tourism, 'smape', 4),
    ('Weather-96', weather, 'mse', 96),
]):
    max_r = df['search_round'].max()
    r3 = df[(df['search_round'] == max_r) & (~df['diverged_flag'])]
    
    t3_vals = r3[r3['trend_thetas_dim_cfg'] == 3][metric].values
    t5_vals = r3[r3['trend_thetas_dim_cfg'] == 5][metric].values
    
    # Box plot
    bp = ax.boxplot([t3_vals, t5_vals], labels=['ttd=3', 'ttd=5'],
                     patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor(TTD_COLORS[3])
    bp['boxes'][1].set_facecolor(TTD_COLORS[5])
    for box in bp['boxes']:
        box.set_alpha(0.7)
    
    # Stats
    u, p = mannwhitneyu(t3_vals, t5_vals, alternative='two-sided')
    winner = 'ttd=3' if np.mean(t3_vals) < np.mean(t5_vals) else 'ttd=5'
    sig = 'SIGNIFICANT' if p < 0.05 else 'not significant'
    
    ax.set_title(f'{ds_name}\n(horizon={horizon})', fontweight='bold')
    ax.set_ylabel(metric.upper())
    
    color = 'green' if p < 0.05 else 'gray'
    ax.text(0.5, 0.92, f'{winner} better\np={p:.4f} ({sig})',
            transform=ax.transAxes, ha='center', va='top', fontsize=9,
            color=color, fontweight='bold')
    
    # Mean annotations
    for i, (vals, ttd) in enumerate([(t3_vals, 3), (t5_vals, 5)]):
        ax.text(i + 1, np.mean(vals), f'{np.mean(vals):.1f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    results_summary.append({
        'dataset': ds_name, 'horizon': horizon,
        'ttd3_mean': np.mean(t3_vals), 'ttd5_mean': np.mean(t5_vals),
        'winner': winner, 'p_value': p
    })

fig.suptitle('Trend Thetas Dim Effect Reverses by Forecast Horizon',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ttd_reversal.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
ttd_df = pd.DataFrame(results_summary)
display(ttd_df.style.format({
    'ttd3_mean': '{:.3f}', 'ttd5_mean': '{:.3f}', 'p_value': '{:.4f}'
}))

**Interpretation:** This is the most actionable finding in the study. The trend polynomial degree preference **reverses by forecast horizon**:

- **Short horizons (H=4, 6):** ttd=3 (quadratic) is better. Higher polynomials overfit the limited trend signal, taking capacity away from the wavelet stack.
- **Long horizons (H=96):** ttd=5 (quartic) is significantly better. The longer forecast benefits from capturing more complex trend shapes.
- **M4-Yearly (H=6):** No significant difference at R3 (p=0.16), but the R1 data strongly favored ttd=3. This indicates ttd=5 simply converges more slowly on short horizons.

This overturns the prior skill recommendation of "always use ttd=3" -- it was based on R1 data that mixed convergence speed with final quality.

**Rule of thumb:** Use ttd=3 for H <= ~10, ttd=5 for H >= ~50. The crossover zone (10-50) has not been tested.

---

## 6. Traffic-96 Divergence Analysis

Traffic-96 showed catastrophic failure. Let us examine divergence patterns by wavelet family and configuration.

## Note: Traffic Failure Root Cause Identified

**All traffic failure results in this notebook were produced with backcast_length=192 (L=2H, forecast_multiplier=2).**

Subsequent study (AsymWavelet Diagnostic, 2026-03-08) using backcast_length=480 (L=5H, forecast_multiplier=5)
achieved 80-100% convergence across multiple block types. The failures shown below are a consequence
of insufficient backcast horizon, not architectural incompatibility.

Use `forecast_multiplier=5` (or `backcast_length=480`) for Traffic-96 experiments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- By wavelet family ---
ax = axes[0]
traf_by_wf = traffic.groupby('wavelet_name').agg(
    total=('smape', 'count'),
    diverged=('diverged_flag', 'sum'),
).reset_index()
traf_by_wf['converged'] = traf_by_wf['total'] - traf_by_wf['diverged']
traf_by_wf['div_rate'] = traf_by_wf['diverged'] / traf_by_wf['total'] * 100
traf_by_wf = traf_by_wf.sort_values('div_rate', ascending=False)

colors = [FAMILY_COLORS.get(n, '#999') for n in traf_by_wf['wavelet_name']]
bars = ax.barh(range(len(traf_by_wf)), traf_by_wf['div_rate'],
               color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(traf_by_wf)))
ax.set_yticklabels(traf_by_wf['wavelet_name'])
ax.set_xlabel('Divergence Rate (%)')
ax.set_title('Traffic-96: Divergence by Wavelet Family', fontweight='bold')
ax.axvline(x=86.2, color='red', linestyle='--', alpha=0.5, label='Overall: 86.2%')
ax.legend(fontsize=9)

# Annotate converged/total
for i, (_, row) in enumerate(traf_by_wf.iterrows()):
    ax.text(row['div_rate'] + 1, i, f"{row['converged']:.0f}/{row['total']:.0f} conv.",
            va='center', fontsize=8)

# --- By bd_label ---
ax = axes[1]
traf_by_bd = traffic.groupby('bd_label').agg(
    total=('smape', 'count'),
    diverged=('diverged_flag', 'sum'),
).reset_index()
traf_by_bd['div_rate'] = traf_by_bd['diverged'] / traf_by_bd['total'] * 100
traf_by_bd = traf_by_bd.sort_values('div_rate', ascending=False)

colors = [BD_COLORS.get(n, '#999') for n in traf_by_bd['bd_label']]
bars = ax.barh(range(len(traf_by_bd)), traf_by_bd['div_rate'],
               color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(traf_by_bd)))
ax.set_yticklabels(traf_by_bd['bd_label'])
ax.set_xlabel('Divergence Rate (%)')
ax.set_title('Traffic-96: Divergence by Basis Dim Label', fontweight='bold')
ax.axvline(x=86.2, color='red', linestyle='--', alpha=0.5)

for i, (_, row) in enumerate(traf_by_bd.iterrows()):
    c = row['total'] - row['diverged']
    ax.text(row['div_rate'] + 1, i, f"{c:.0f}/{row['total']:.0f} conv.",
            va='center', fontsize=8)

plt.tight_layout()
plt.savefig('traffic_divergence.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
total = len(traffic)
div = traffic['diverged_flag'].sum()
print(f"\nTraffic-96 overall: {div}/{total} diverged ({div/total*100:.1f}%)")
print(f"Only {total - div} runs showed any learning. Architecture is NOT VIABLE for Traffic-96.")

**Interpretation:** Traffic-96 is a total failure for TrendAELG + WaveletV3AELG. DB20 diverged in 100% of runs; Haar in 96%. Only DB2, DB3, and DB4 had *any* converged runs, and even those had 75-83% divergence. No basis_dim label rescued the architecture.

The likely root cause is the extreme input length (L=480 with forecast_multiplier=5) combined with the AE bottleneck (latent_dim=16). The bottleneck cannot preserve enough information to reconstruct 480-step backcasts with only 16 latent dimensions.

**Do NOT use this architecture for Traffic-96.** Alternatives to explore: non-AE Trend+WaveletV3, larger latent_dim (32+), or entirely different block families (Generic, BottleneckGeneric).

---

## 7. Successive Halving Reliability: Do R1 Rankings Predict R3 Winners?

If early-round rankings (R1, 10 epochs) perfectly predicted final performance (R3, 50 epochs), we could run fewer experiments. But convergence speed differences between configurations can mislead early rankings. Let us measure how well R1 ranks predict R3 ranks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (ds_name, df, metric) in zip(axes, [
    ('M4-Yearly', m4, 'smape'),
    ('Weather-96', weather, 'mse'),
]):
    # Get R1 and R3 mean metrics per config
    r1 = df[(df['search_round'] == 1) & (~df['diverged_flag'])]
    max_r = df['search_round'].max()
    r3 = df[(df['search_round'] == max_r) & (~df['diverged_flag'])]
    
    r1_means = r1.groupby('config_name')[metric].mean()
    r3_means = r3.groupby('config_name')[metric].mean()
    
    # Only configs present in both rounds
    common = r1_means.index.intersection(r3_means.index)
    r1_ranks = r1_means[common].rank()
    r3_ranks = r3_means[common].rank()
    
    # Spearman correlation
    rho, p = spearmanr(r1_ranks, r3_ranks)
    
    # Scatter plot
    ax.scatter(r1_ranks, r3_ranks, alpha=0.5, s=30, color='steelblue', edgecolors='white', linewidths=0.5)
    
    # Perfect prediction line
    max_rank = max(r1_ranks.max(), r3_ranks.max())
    ax.plot([1, max_rank], [1, max_rank], 'r--', alpha=0.4, label='Perfect prediction')
    
    # Highlight top-10 R3 winners
    r3_top10 = r3_ranks.nsmallest(10).index
    for cfg in r3_top10:
        ax.scatter(r1_ranks[cfg], r3_ranks[cfg], color='red', s=60, zorder=5, edgecolors='black', linewidths=0.8)
    
    ax.set_xlabel(f'R1 Rank ({metric.upper()}, 10 epochs)')
    ax.set_ylabel(f'R3 Rank ({metric.upper()}, {max_r*10 + 30} epochs)')
    ax.set_title(f'{ds_name}\nSpearman rho={rho:.3f}, p={p:.1e}', fontweight='bold')
    ax.legend(fontsize=9)
    
    # Annotate: how many R3 top-10 were in R1 top-25?
    r1_top25 = r1_ranks.nsmallest(25).index
    overlap = len(set(r3_top10) & set(r1_top25))
    ax.text(0.05, 0.95, f'R3 top-10 in R1 top-25:\n{overlap}/10 ({overlap*10}%)',
            transform=ax.transAxes, va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

fig.suptitle('R1 vs R3 Rank Correlation: Does Early Training Predict Final Quality?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('successive_halving_reliability.png', dpi=150, bbox_inches='tight')
plt.show()

# Additional analysis: which configs moved the most?
print("\n=== Biggest R1 -> R3 rank improvements (configs that benefited most from longer training) ===")
for ds_name, df, metric in [('M4-Yearly', m4, 'smape'), ('Weather-96', weather, 'mse')]:
    r1 = df[(df['search_round'] == 1) & (~df['diverged_flag'])]
    max_r = df['search_round'].max()
    r3 = df[(df['search_round'] == max_r) & (~df['diverged_flag'])]
    r1_means = r1.groupby('config_name')[metric].mean()
    r3_means = r3.groupby('config_name')[metric].mean()
    common = r1_means.index.intersection(r3_means.index)
    r1_ranks = r1_means[common].rank()
    r3_ranks = r3_means[common].rank()
    improvement = r1_ranks - r3_ranks  # positive = improved from R1 to R3
    print(f"\n{ds_name}:")
    top_improvers = improvement.nlargest(5)
    for cfg, delta in top_improvers.items():
        print(f"  {cfg}: R1 rank {r1_ranks[cfg]:.0f} -> R3 rank {r3_ranks[cfg]:.0f} (improved {delta:.0f} places)")

**Interpretation:** R1 rankings are moderately predictive of R3 rankings but far from perfect. The Spearman correlation is positive, meaning early training provides useful signal, but substantial re-ordering occurs as training continues.

Key observations:
- **Most R3 top-10 configs were in the R1 top-25**, validating successive halving as a search strategy -- it keeps the right configs alive. But several top R3 performers ranked poorly at R1, meaning aggressive early elimination (keep_fraction < 0.4) risks losing eventual winners.
- **Configs with larger basis_dim tend to rank well early** (convergence speed advantage) but do not necessarily win at convergence. This is the same artifact seen in Section 4.
- **The biggest rank improvers** tend to be configs with `eq_fcast` (smaller basis_dim) that start slowly but converge to equivalent or better quality.

**Practical takeaway:** Keep at least 50% of configs from R1 to R2. The current study design (112 -> 75 -> 50) is appropriately conservative.

---

## 8. Summary and Recommendations

### Best Configurations per Dataset (R3 Converged)

| Dataset | Best Config | Primary Metric | Notes |
|---|---|---|---|
| M4-Yearly | Symlet20_eq_fcast_ttd3_ld16 | SMAPE=13.438, OWA=0.795 | Replicates V1 AELG result exactly |
| Tourism-Yearly | Coif1_eq_fcast_ttd3_ld16 | SMAPE=20.930 | Replicates V1 AELG result exactly |
| Weather-96 | Symlet20_eq_fcast_ttd5_ld16 | MSE=2070.61 | 3 seeds only, high variance |
| Traffic-96 | **NOT VIABLE** | 86.2% divergence | Do not use this architecture |

### Key Findings

1. **Wavelet family selection is horizon-dependent** but Symlet20 is the universal safe choice (avg rank 2.3/14 across datasets).
2. **basis_dim = forecast_length** (`eq_fcast`) is the optimal or co-optimal setting on all viable datasets.
3. **trend_thetas_dim must match forecast horizon**: ttd=3 for short horizons (H <= ~10), ttd=5 for long horizons (H >= ~50). This overturns the prior "always ttd=3" recommendation.
4. **R1 rankings are useful but imperfect** guides for successive halving. Keep >= 50% of configs per round.
5. **Traffic-96 requires a fundamentally different approach** -- the AE bottleneck with latent_dim=16 cannot handle 480-step inputs.
6. **The non-AE Trend+WaveletV3 architecture** (SMAPE=13.410 on M4-Yearly) still slightly outperforms the AELG variant (13.438). The AE bottleneck adds complexity without clear benefit on M4.

### What to Test Next
- Extend Weather-96 best config to 10 seeds for variance reduction
- Test M4 other periods (Quarterly, Monthly) to map the ttd crossover point
- Traffic recovery: try non-AE Trend+WaveletV3, latent_dim=32/64, or skip connections
- Compare AELG vs non-AE on Weather to determine if AE bottleneck helps on longer horizons